# Q-Mini — One-Click Initial Model Builder

**यह नोटबुक एक ही code cell से पूरा initial build करती है।**

1. Google Drive की अनुमति दें।
2. नीचे के **एकमात्र code cell** को Run करें (या Runtime → Run all)।
3. यह GitHub से latest source अपने-आप डाउनलोड करेगा, आवश्यक runtime जाँचेगा, 512 context वाला untrained Q2-v2 model बनाएगा, उसकी जाँच करेगा और पूरा project Google Drive में सेव करेगा।

किसी Python cell को अलग से चलाने, GitHub code हाथ से कॉपी करने या dataset upload करने की जरूरत initial build के लिए नहीं है।

**ध्यान दें:** यह केवल initial random model बनाता है। वास्तविक training शुरू नहीं होती। बाद में training के लिए tokenizer और उपयोग की अनुमति वाला train/validation text data चाहिए होगा।

In [ ]:
# Q-MINI ONE-CLICK BUILDER
# Run this ONE cell. It downloads the latest GitHub source, builds and verifies
# the untrained Q2-v2 initial model, and saves the complete project to Google Drive.

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid

# --------------------------- Fixed project settings ---------------------------
MODEL_CONFIG = "configs/qmini_light_0.7m.json"
REPOSITORY_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
DRIVE_SUBFOLDER = "QMini-Light-0.7M/initial_brains"
EXPECTED_CONTEXT = 512

def run_checked(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def require_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Required build file is missing: {path}")
    return path

print("\n[1/6] Checking Colab and Google Drive...")
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Drive mount failed. Please allow the Google Drive permission request and run this cell again.")

DRIVE_PARENT = DRIVE_ROOT / DRIVE_SUBFOLDER
DRIVE_PARENT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
REPO_DIR = Path("/content") / f"Q-Mini-{RUN_ID}"
LOCAL_BUILD_DIR = Path("/content") / f"QMini-initial-{RUN_ID}"
FINAL_DIR = DRIVE_PARENT / f"QMini-qmini_light_0.7m-Q2v2-initial-{RUN_ID}"
STAGING_DIR = DRIVE_PARENT / f".{FINAL_DIR.name}.staging"

print("\n[2/6] Downloading the latest project directly from GitHub...")
run_checked(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, REPO_DIR])
SOURCE_REVISION = run_checked(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR
)
SOURCE_REVISION = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True
).strip()
if not (REPO_DIR / "qmini" / "__init__.py").is_file():
    raise RuntimeError("The GitHub download does not contain the expected qmini Python package.")
if not (REPO_DIR / MODEL_CONFIG).is_file():
    raise RuntimeError(f"Model configuration was not found in the downloaded project: {MODEL_CONFIG}")

# Make the freshly downloaded repository importable in THIS cell and in any
# child process. This avoids relying on a previous pip install or notebook cell.
sys.path.insert(0, str(REPO_DIR))
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("GitHub revision:", SOURCE_REVISION)
print("Source folder:", REPO_DIR)

print("\n[3/6] Checking the runtime and model configuration...")
try:
    import torch
    import numpy as np
except ImportError as exc:
    raise RuntimeError(
        "This Colab runtime is missing PyTorch or NumPy. Please use a standard "
        "Google Colab Python runtime, then run this same cell again. No project "
        "files need to be uploaded."
    ) from exc

from qmini.config import QMiniConfig
from qmini.initial_build import build_initial_brain
from qmini.export import verify_packed_export

cfg = QMiniConfig.from_json(REPO_DIR / MODEL_CONFIG)
cfg.validate()
if cfg.max_context != EXPECTED_CONTEXT:
    raise RuntimeError(f"This project is configured for context {cfg.max_context}; expected {EXPECTED_CONTEXT}.")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The initial build runs on CPU; GPU training is not started.")
print("Model parameters:", f"{cfg.estimated_parameter_count():,}")
print("Context length:", cfg.max_context)

print("\n[4/6] Building the initial model (training is NOT started)...")
BUILD_STARTED_UTC = datetime.now(timezone.utc).isoformat()
build_started = time.monotonic()
build_manifest = build_initial_brain(
    cfg,
    LOCAL_BUILD_DIR,
    source_revision=SOURCE_REVISION,
)
BUILD_SECONDS = round(time.monotonic() - build_started, 2)

PACKED_PATH = require_file(LOCAL_BUILD_DIR / "model.qmini.npz")
MANIFEST_PATH = require_file(LOCAL_BUILD_DIR / "model_manifest.json")
CONFIG_OUT_PATH = require_file(LOCAL_BUILD_DIR / "config.json")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert manifest.get("format") == "qmini-initial-brain-v1"
assert manifest.get("status") == "untrained_initialization"
assert manifest.get("training_steps") == 0
assert manifest.get("optimizer_steps") == 0
assert manifest.get("tokens_seen") == 0
assert manifest.get("training_started") is False
assert manifest.get("matrix_weight_format") == "qmini-packed-v2"
assert manifest.get("matrix_weight_bits") == 2
assert manifest.get("parameters") == cfg.estimated_parameter_count()

LOCAL_SHA256 = sha256_file(PACKED_PATH)
assert LOCAL_SHA256 == manifest.get("packed_export_sha256"), "Local model file checksum did not match its manifest."
LOCAL_VERIFY = verify_packed_export(PACKED_PATH)
assert LOCAL_VERIFY.get("format") == "qmini-packed-v2"
assert LOCAL_VERIFY.get("q2_version") == 2
assert LOCAL_VERIFY.get("parameter_count") == cfg.estimated_parameter_count()

print("Model built and verified.")
print("Packed model size:", f"{PACKED_PATH.stat().st_size:,}", "bytes")
print("SHA-256:", LOCAL_SHA256)
print("Build time:", BUILD_SECONDS, "seconds")

print("\n[5/6] Saving the model and complete project to Google Drive...")
if FINAL_DIR.exists() or STAGING_DIR.exists():
    raise FileExistsError("A Drive output folder with this unique run ID already exists. Run the cell again to create a fresh run.")

STAGING_DIR.mkdir(parents=False, exist_ok=False)
for filename in ("config.json", "model.qmini.npz", "model_manifest.json"):
    shutil.copy2(LOCAL_BUILD_DIR / filename, STAGING_DIR / filename)

DRIVE_PACKED_PATH = require_file(STAGING_DIR / "model.qmini.npz")
DRIVE_SHA256 = sha256_file(DRIVE_PACKED_PATH)
if DRIVE_SHA256 != LOCAL_SHA256:
    raise RuntimeError("Google Drive copy verification failed: the file checksum changed during copying.")

DRIVE_VERIFY = verify_packed_export(DRIVE_PACKED_PATH)
if DRIVE_VERIFY.get("parameter_count") != cfg.estimated_parameter_count():
    raise RuntimeError("The Google Drive model copy has an unexpected parameter count.")

drive_manifest = json.loads((STAGING_DIR / "model_manifest.json").read_text(encoding="utf-8"))
if drive_manifest.get("packed_export_sha256") != DRIVE_SHA256:
    raise RuntimeError("The Google Drive model manifest checksum does not match the copied model.")

# Complete source checkout, including scripts, tests, configs and documentation.
PROJECT_SOURCE_DIR = STAGING_DIR / "project_source"
shutil.copytree(
    REPO_DIR,
    PROJECT_SOURCE_DIR,
    ignore=shutil.ignore_patterns(
        ".git", "__pycache__", "*.pyc", ".pytest_cache", ".venv", "venv",
        ".mypy_cache", ".ruff_cache",
    ),
)

# Matching model/training/checkpoint Python source, kept disabled.
RUNTIME_DIR = STAGING_DIR / "training_runtime"
RUNTIME_DIR.mkdir()
shutil.copytree(
    REPO_DIR / "qmini",
    RUNTIME_DIR / "qmini",
    ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.so", "*.pyd"),
)
shutil.copy2(REPO_DIR / "pyproject.toml", RUNTIME_DIR / "pyproject.toml")

TRAINING_DEFAULTS = {
    "format": "qmini-training-defaults-v1",
    "enabled": False,
    "sequence_length": EXPECTED_CONTEXT,
    "batch_size": 1,
    "gradient_accumulation": 8,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,
    "checkpoint_every_optimizer_steps": 10,
    "evaluation_every_optimizer_steps": 100,
    "total_optimizer_steps": 20000,
    "minimum_free_gpu_gib": 8.0,
}
(RUNTIME_DIR / "TRAINING_DEFAULTS.json").write_text(
    json.dumps(TRAINING_DEFAULTS, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(RUNTIME_DIR / "README.md").write_text(
    "# Q-Mini training source bundle\n\n"
    "This contains the qmini source matching the initial model's GitHub revision. "
    "Training is disabled and has not been run. A tokenizer and matching prepared "
    "train/validation data are still required before actual model training.\n",
    encoding="utf-8",
)

runtime_source_files = []
for source_file in sorted(
    p for p in RUNTIME_DIR.rglob("*")
    if p.is_file() and p.name != "TRAINING_RUNTIME.json"
):
    runtime_source_files.append({
        "path": source_file.relative_to(RUNTIME_DIR).as_posix(),
        "sha256": sha256_file(source_file),
        "size_bytes": source_file.stat().st_size,
    })

RUNTIME_MANIFEST = {
    "format": "qmini-training-runtime-v1",
    "enabled": False,
    "requires_manual_activation": True,
    "entrypoint": "qmini.train.train_model",
    "checkpoint_backend": "qmini.checkpointing",
    "source_revision": SOURCE_REVISION,
    "source_files": runtime_source_files,
}
RUNTIME_MANIFEST_PATH = RUNTIME_DIR / "TRAINING_RUNTIME.json"
RUNTIME_MANIFEST_PATH.write_text(
    json.dumps(RUNTIME_MANIFEST, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

STATE_DIR = STAGING_DIR / "training_state"
STATE_DIR.mkdir()
(STATE_DIR / "training_progress.json").write_text(
    json.dumps({
        "format": "qmini-training-progress-v1",
        "status": "not_started",
        "current_step": 0,
        "target_steps": None,
        "progress_percent": 0.0,
        "tokens_seen": 0,
        "updated_at": datetime.now(timezone.utc).isoformat(),
    }, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

drive_manifest["training_runtime_included"] = True
drive_manifest["training_runtime_enabled"] = False
drive_manifest["training_runtime_manifest_sha256"] = sha256_file(RUNTIME_MANIFEST_PATH)
(STAGING_DIR / "model_manifest.json").write_text(
    json.dumps(drive_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

project_file_count = sum(1 for p in PROJECT_SOURCE_DIR.rglob("*") if p.is_file())
BUILD_REPORT = {
    "format": "qmini-colab-build-report-v1",
    "run_id": RUN_ID,
    "repository": REPOSITORY_URL,
    "source_revision": SOURCE_REVISION,
    "config": MODEL_CONFIG,
    "parameters": cfg.estimated_parameter_count(),
    "context_length": cfg.max_context,
    "architecture": {
        "causal_convolution": True,
        "diagonal_state_space": bool(cfg.use_state_space),
        "sparse_top1_moe": True,
        "transformer_self_attention": False,
        "qkv_attention": False,
    },
    "artifact": {
        "filename": "model.qmini.npz",
        "format": "qmini-packed-v2",
        "matrix_weight_bits": 2,
        "bytes": DRIVE_PACKED_PATH.stat().st_size,
        "sha256": DRIVE_SHA256,
        "verified_after_drive_copy": True,
        "verification": DRIVE_VERIFY,
    },
    "training": {
        "started": False,
        "training_steps": 0,
        "optimizer_steps": 0,
        "tokens_seen": 0,
        "tokenizer_included": False,
        "dataset_prepared": False,
    },
    "runtime": {
        "python": sys.version.split()[0],
        "pytorch": torch.__version__,
        "numpy": np.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    "build_started_utc": BUILD_STARTED_UTC,
    "build_completed_utc": datetime.now(timezone.utc).isoformat(),
    "build_seconds": BUILD_SECONDS,
    "project_source_directory": "project_source/",
    "project_source_file_count": project_file_count,
    "training_runtime_directory": "training_runtime/",
    "training_runtime_source_files": len(runtime_source_files),
}
(STAGING_DIR / "BUILD_REPORT.json").write_text(
    json.dumps(BUILD_REPORT, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
    encoding="utf-8",
)

# Publish the folder only after all copy checks and reports have succeeded.
STAGING_DIR.rename(FINAL_DIR)

print("\n[6/6] ALL CHECKS PASSED — Q-MINI INITIAL BUILD COMPLETE")
print("Google Drive folder:", FINAL_DIR)
print("Model file:", FINAL_DIR / "model.qmini.npz")
print("Build report:", FINAL_DIR / "BUILD_REPORT.json")
print("Project source:", FINAL_DIR / "project_source")
print("Training code bundle:", FINAL_DIR / "training_runtime")
print("Source files saved:", project_file_count)
print("Training/checkpoint source files saved:", len(runtime_source_files))
print("Parameters:", f"{cfg.estimated_parameter_count():,}")
print("Context length:", cfg.max_context)
print("Training steps: 0")
print("TRAINING STARTED: NO")
print("\nImportant: this is a verified random initialization, not a trained assistant. Training has not started.")
print("A tokenizer and matching permitted training data are required for the later training stage.")
